# 04. Sampling a saved checkpoint

A checkpoint outlives the session that saved it. Notebook 02 ended by publishing its adapter
as `banking77-final`, and closing that session did not delete it. This notebook finds that
checkpoint again from a fresh process and serves it.

It takes two sessions, and the difference between them is the point:

1. **A session with no models**, which holds no GPUs and returns immediately. It is only a
   handle on your account, used here to list what you have saved.
2. **A session that declares the checkpoint's base model**, which provisions a sampler for
   it. The checkpoint is a LoRA adapter, so it is served on top of the base model it was
   trained from.

Run notebook 02 first, or have a checkpoint of your own saved with
`save_weights_for_sampler`.

## Setup

Two things to do: install the SDK, and authenticate with an Intelligence API key from the
[Crusoe Console](https://console.crusoecloud.com/).

`crusoe-mill` is in limited preview and not on a public package index yet, so it ships next
to this notebook as a wheel in `sdk/`. `%pip` installs it into the interpreter running **this
notebook**, which is what keeps the import below from landing in a different environment than
the kernel.

The two lines above it are only there because `uv`-created virtual environments ship without
`pip`, and `%pip` needs it. `ensurepip` is in the standard library and carries its own copy,
so nothing extra is downloaded.

In [1]:
try:
    import pip
except ImportError:
    import ensurepip
    ensurepip.bootstrap()

%pip install --quiet --upgrade -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


Now the credentials. The cell below reads `CRUSOE_API_KEY` from your environment and prompts
for it if it is not set, so the key never gets saved into this notebook file.

`CRUSOE_BASE_URL` has to be set explicitly: the SDK's built-in default endpoint is not the one
serving this preview.

In [2]:
import os
from getpass import getpass

if not os.environ.get("CRUSOE_API_KEY"):
    os.environ["CRUSOE_API_KEY"] = getpass("Crusoe API key: ")

os.environ.setdefault("CRUSOE_BASE_URL", "https://api.intelligence.crusoecloud.com")

print("endpoint:", os.environ["CRUSOE_BASE_URL"])

endpoint: https://api.intelligence.crusoecloud.com


## 1. List your checkpoints

A `ServiceClient` built without a `ServiceConfig` declares no models, so there is nothing to
provision and it returns at once. Its REST client reads your account: checkpoints from every
session you have run, not only this one.

In [3]:
from crusoe_mill import ServiceClient

catalog = ServiceClient()
rest = catalog.create_rest_client()

checkpoints = rest.list_user_checkpoints().result().checkpoints
checkpoints.sort(key=lambda checkpoint: checkpoint.time, reverse=True)

for checkpoint in checkpoints:
    print(f"{checkpoint.time:%Y-%m-%d %H:%M}  {checkpoint.checkpoint_type:8}  {checkpoint.mill_path}")

2026-09-29 18:38  sampler   mill://msess_bd210cb66c56465396a6ae1221b68829/sampler_weights/banking77-final
2026-09-29 01:19  sampler   mill://msess_da2d573a7c26447f96f95cf791a689d6/sampler_weights/final
2026-09-29 00:27  sampler   mill://msess_8062926038774f81a44997dc22f950dd/sampler_weights/final


Only `sampler` checkpoints can be served; the other kind, from `save_state`, is for resuming
training. Pick notebook 02's adapter if it is there, otherwise the newest one, then ask the
service what it was trained from.

In [4]:
CHECKPOINT_NAME = "banking77-final"

servable = [c for c in checkpoints if c.checkpoint_type == "sampler"]
if not servable:
    raise RuntimeError("No sampler checkpoints found. Run notebook 02 first, "
                       "or save one of your own with save_weights_for_sampler.")
checkpoint = next((c for c in servable if c.checkpoint_id.endswith(CHECKPOINT_NAME)), servable[0])
if not checkpoint.checkpoint_id.endswith(CHECKPOINT_NAME):
    print(f"{CHECKPOINT_NAME!r} not found; using the newest checkpoint instead")

info = rest.get_weights_info_by_tinker_path(checkpoint.mill_path).result()
print("checkpoint :", checkpoint.mill_path)
print("base model :", info.base_model)
print("LoRA rank  :", info.lora_rank)

catalog.close()

checkpoint : mill://msess_bd210cb66c56465396a6ae1221b68829/sampler_weights/banking77-final
base model : Qwen/Qwen3.5-9B
LoRA rank  : 16


## 2. Open a session that can serve it

Declare the checkpoint's base model as a sampling row. As in the earlier notebooks,
constructing the `ServiceClient` blocks until the sampler is ready. The same session can
then serve both the base model and the adapter, since the adapter is loaded on top of it.

In [5]:
from crusoe_mill import SamplingBaseModelConfig, ServiceConfig

client = ServiceClient(
    service_config=ServiceConfig(
        sampling_client_base_models=[SamplingBaseModelConfig(base_model=info.base_model)],
    ),
)

adapter_sampler = client.create_sampling_client(model_path=checkpoint.mill_path)
base_sampler = client.create_sampling_client(base_model=info.base_model)
tokenizer = adapter_sampler.get_tokenizer()
print("serving", checkpoint.mill_path)

[2026-09-29 22:26:43 crusoe_mill._progress INFO] tinker session msess_963cf0afc43f40a2a6cad0bfd12fff3c created; provisioning Qwen/Qwen3.5-9B


[2026-09-29 22:26:43 crusoe_mill.service_client INFO] ServiceClient initialized for session msess_963cf0afc43f40a2a6cad0bfd12fff3c


[2026-09-29 22:26:44 crusoe_mill._progress INFO] provisioning — 2s elapsed (session msess_963cf0afc43f40a2a6cad0bfd12fff3c; ^C releases it)


[2026-09-29 22:27:54 crusoe_mill._progress INFO] provisioning — 72s elapsed (session msess_963cf0afc43f40a2a6cad0bfd12fff3c; ^C releases it)


[2026-09-29 22:28:56 crusoe_mill._progress INFO] provisioning — 135s elapsed (session msess_963cf0afc43f40a2a6cad0bfd12fff3c; ^C releases it)


PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


serving mill://msess_bd210cb66c56465396a6ae1221b68829/sampler_weights/banking77-final


## 3. Sample from it

Notebook 02's held-out test split, answered greedily by the base model and by the adapter.
The adapter was trained to reply with exactly one intent name, so it should score higher,
and the rows where the two disagree show what the training changed. If you picked a
checkpoint of your own, swap in prompts from its task.

In [6]:
import json
from pathlib import Path

from prepare_data import ensure_banking77
from tinker import types

ensure_banking77()
rows = [json.loads(line) for line in Path("data/banking77/test.jsonl").read_text().splitlines() if line.strip()]
params = types.SamplingParams(max_tokens=16, temperature=0.0)


def answers(sampler):
    """Greedy answer for every row, submitted together and collected in order."""
    futures = []
    for row in rows:
        prompt_ids = tokenizer.apply_chat_template(
            [m for m in row["messages"] if m["role"] != "assistant"],
            add_generation_prompt=True, tokenize=True, return_dict=False, enable_thinking=False,
        )
        futures.append(sampler.sample(
            prompt=types.ModelInput.from_ints(prompt_ids), sampling_params=params, num_samples=1,
        ))
    return [tokenizer.decode(f.result().sequences[0].tokens, skip_special_tokens=True).strip()
            for f in futures]


gold = [row["messages"][-1]["content"] for row in rows]
base_answers, adapter_answers = answers(base_sampler), answers(adapter_sampler)

for label, predicted in (("base", base_answers), ("adapter", adapter_answers)):
    correct = sum(p == g for p, g in zip(predicted, gold))
    print(f"{label:8} {correct / len(rows):6.1%}  ({correct}/{len(rows)})")

fixed = [i for i in range(len(rows)) if base_answers[i] != gold[i] and adapter_answers[i] == gold[i]]
print(f"\n{len(fixed)} rows the base model gets wrong and the adapter gets right, for example:\n")
for i in fixed[:4]:
    print(rows[i]["messages"][1]["content"])
    print(f"  base    : {base_answers[i]}")
    print(f"  adapter : {adapter_answers[i]}\n")

banking77 already built and verified


base      72.7%  (168/231)
adapter   83.5%  (193/231)

33 rows the base model gets wrong and the adapter gets right, for example:

I would appreciate that someone let me know when there is an extra charge for payments. I was checking the app earlier and saying a charge on one of my payments that was additional that no one made me aware of before.
  base    : extra_charge_on_statement
  adapter : card_payment_fee_charged

I want to add money but don't know what currencies you accept, can you tell me?
  base    : fiat_currency_support
  adapter : supported_cards_and_currencies

Where is the virtual card located?
  base    : get_virtual_card
  adapter : getting_virtual_card

What do I do if it says my card payment has been cancelled?
  base    : cancel_transfer
  adapter : reverted_card_payment?



## Close the session

The second session holds a sampler, so close it when you are done. The first one held
nothing and is already closed. Closing a session never deletes its checkpoints: they stay in
your account, and you can serve them again from any later session the same way.

In [7]:
client.close()
print("session closed")

session closed
